# Lab Assignment 3 — Task 2

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` — a non-terminal produces two non-terminals.
2. `A -> 'word'` — a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [6]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [7]:
import math
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser

# Define CNF grammar (converting VP -> V NP PP into binary rules)
cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

# Build CNF PCFG from the provided string
pcfg = PCFG.fromstring(cnf_grammar_str)

# Parse with ViterbiParser (most probable parse using PCFG)
parser = ViterbiParser(pcfg)
words = sentence.split()
parses = list(parser.parse(words))

if not parses:
    print('No parse found by ViterbiParser')
else:
    best_tree = parses[0]
    print('Most probable parse (Viterbi):')
    print(best_tree)
    
    # Extract rules from the parse tree and calculate total probability
    print('\nRules used in the parse tree:')
    total_prob = 1.0
    for prod in best_tree.productions():
        # prod is a Production object; get its string representation
        prod_str = str(prod)
        print('  ', prod_str)
        
        # For PCFG productions, look up the probability in the grammar
        lhs_str = prod.lhs().symbol()
        rhs_list = [r.symbol() if hasattr(r, 'symbol') else r for r in prod.rhs()]
        
        # Find matching production in PCFG
        for pcfg_prod in pcfg.productions():
            if pcfg_prod.lhs().symbol() == lhs_str:
                pcfg_rhs = [r.symbol() if hasattr(r, 'symbol') else r for r in pcfg_prod.rhs()]
                if pcfg_rhs == rhs_list:
                    total_prob *= float(pcfg_prod.prob())
                    break
    
    print('\nTotal probability of this parse (product of rule probs):', total_prob)

# Helper to lookup production probability in the grammar
def get_prod_prob(grammar, lhs, rhs_tuple):
    """Look up probability of a production LHS -> RHS in the grammar."""
    for prod in grammar.productions():
        if prod.lhs().symbol() == lhs:
            rhs = prod.rhs()
            # normalize rhs to tuple of symbols/strings
            match = True
            if len(rhs) != len(rhs_tuple):
                continue
            for a, b in zip(rhs, rhs_tuple):
                if hasattr(a, 'symbol'):  # Nonterminal
                    if a.symbol() != b:
                        match = False
                        break
                else:  # terminal (string)
                    if a != b:
                        match = False
                        break
            if match:
                return float(prod.prob())
    return 0.0

# Manually compute the two competing interpretations' probabilities
print('\n' + '='*60)
print('ANALYSIS OF COMPETING INTERPRETATIONS')
print('='*60)

# Interpretation A: PP attaches to the object NP ("the man with a telescope")
print('\nInterpretation A: "I SAW [the man WITH a telescope]"')
print('  (PP attaches to object NP)')
prob_attach_np = 1.0
probs_a = []

p = get_prod_prob(pcfg, 'S', ('NP', 'VP'))
prob_attach_np *= p
probs_a.append(('S -> NP VP', p))

p = get_prod_prob(pcfg, 'NP', ('I',))
prob_attach_np *= p
probs_a.append(("NP -> 'I'", p))

p = get_prod_prob(pcfg, 'VP', ('V', 'NP'))
prob_attach_np *= p
probs_a.append(('VP -> V NP', p))

p = get_prod_prob(pcfg, 'V', ('saw',))
prob_attach_np *= p
probs_a.append(("V -> 'saw'", p))

# object NP with PP attached
p = get_prod_prob(pcfg, 'NP', ('NP', 'PP'))
prob_attach_np *= p
probs_a.append(('NP -> NP PP', p))

# inner NP
p = get_prod_prob(pcfg, 'NP', ('Det', 'N'))
prob_attach_np *= p
probs_a.append(('NP -> Det N', p))

p = get_prod_prob(pcfg, 'Det', ('the',))
prob_attach_np *= p
probs_a.append(("Det -> 'the'", p))

p = get_prod_prob(pcfg, 'N', ('man',))
prob_attach_np *= p
probs_a.append(("N -> 'man'", p))

# PP
p = get_prod_prob(pcfg, 'PP', ('P', 'NP'))
prob_attach_np *= p
probs_a.append(('PP -> P NP', p))

p = get_prod_prob(pcfg, 'P', ('with',))
prob_attach_np *= p
probs_a.append(("P -> 'with'", p))

# NP inside PP
p = get_prod_prob(pcfg, 'NP', ('Det', 'N'))
prob_attach_np *= p
probs_a.append(('NP -> Det N', p))

p = get_prod_prob(pcfg, 'Det', ('a',))
prob_attach_np *= p
probs_a.append(("Det -> 'a'", p))

p = get_prod_prob(pcfg, 'N', ('telescope',))
prob_attach_np *= p
probs_a.append(("N -> 'telescope'", p))

for rule, p_val in probs_a:
    print(f'    {rule}: {p_val}')
print(f'  Total: {prob_attach_np}')

# Interpretation B: PP attaches to the VP
print('\nInterpretation B: "I SAW(...) WITH a telescope"')
print('  (PP attaches to VP)')
prob_attach_vp = 1.0
probs_b = []

p = get_prod_prob(pcfg, 'S', ('NP', 'VP'))
prob_attach_vp *= p
probs_b.append(('S -> NP VP', p))

p = get_prod_prob(pcfg, 'NP', ('I',))
prob_attach_vp *= p
probs_b.append(("NP -> 'I'", p))

p = get_prod_prob(pcfg, 'VP', ('V', 'VP_PP'))
prob_attach_vp *= p
probs_b.append(('VP -> V VP_PP', p))

p = get_prod_prob(pcfg, 'V', ('saw',))
prob_attach_vp *= p
probs_b.append(("V -> 'saw'", p))

p = get_prod_prob(pcfg, 'VP_PP', ('NP', 'PP'))
prob_attach_vp *= p
probs_b.append(('VP_PP -> NP PP', p))

# object NP
p = get_prod_prob(pcfg, 'NP', ('Det', 'N'))
prob_attach_vp *= p
probs_b.append(('NP -> Det N', p))

p = get_prod_prob(pcfg, 'Det', ('the',))
prob_attach_vp *= p
probs_b.append(("Det -> 'the'", p))

p = get_prod_prob(pcfg, 'N', ('man',))
prob_attach_vp *= p
probs_b.append(("N -> 'man'", p))

# PP
p = get_prod_prob(pcfg, 'PP', ('P', 'NP'))
prob_attach_vp *= p
probs_b.append(('PP -> P NP', p))

p = get_prod_prob(pcfg, 'P', ('with',))
prob_attach_vp *= p
probs_b.append(("P -> 'with'", p))

# NP inside PP
p = get_prod_prob(pcfg, 'NP', ('Det', 'N'))
prob_attach_vp *= p
probs_b.append(('NP -> Det N', p))

p = get_prod_prob(pcfg, 'Det', ('a',))
prob_attach_vp *= p
probs_b.append(("Det -> 'a'", p))

p = get_prod_prob(pcfg, 'N', ('telescope',))
prob_attach_vp *= p
probs_b.append(("N -> 'telescope'", p))

for rule, p_val in probs_b:
    print(f'    {rule}: {p_val}')
print(f'  Total: {prob_attach_vp}')

print('\n' + '='*60)
print('CONCLUSION:')
print('='*60)
print(f'Interpretation A (NP-attached):  {prob_attach_np}')
print(f'Interpretation B (VP-attached):  {prob_attach_vp}')
print(f'Ratio A/B: {prob_attach_np / prob_attach_vp if prob_attach_vp > 0 else "undefined"}')

if prob_attach_np > prob_attach_vp:
    print('\n✓ Interpretation A is MORE PROBABLE.')
    print('  ViterbiParser should select this interpretation.')
else:
    print('\n✓ Interpretation B is MORE PROBABLE.')
    print('  ViterbiParser should select this interpretation.')


Most probable parse (Viterbi):
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

Rules used in the parse tree:
   S -> NP VP
   NP -> 'I'
   VP -> V NP
   V -> 'saw'
   NP -> NP PP
   NP -> Det N
   Det -> 'the'
   N -> 'man'
   PP -> P NP
   P -> 'with'
   NP -> Det N
   Det -> 'a'
   N -> 'telescope'

Total probability of this parse (product of rule probs): 0.0001512

ANALYSIS OF COMPETING INTERPRETATIONS

Interpretation A: "I SAW [the man WITH a telescope]"
  (PP attaches to object NP)
    S -> NP VP: 1.0
    NP -> 'I': 0.1
    VP -> V NP: 0.7
    V -> 'saw': 1.0
    NP -> NP PP: 0.6
    NP -> Det N: 0.3
    Det -> 'the': 0.8
    N -> 'man': 0.5
    PP -> P NP: 1.0
    P -> 'with': 1.0
    NP -> Det N: 0.3
    Det -> 'a': 0.2
    N -> 'telescope': 0.5
  Total: 0.0001512

Interpretation B: "I SAW(...) WITH a telescope"
  (PP attaches to VP)
    S -> NP VP: 1.0
    NP -> 'I': 0.1
    VP -> V VP_PP: 0.3
 

### Analysis requirement

Do not hard-code the final probability.

Use the parse tree returned by the parser to determine which rules were selected. Then calculate the probability of the competing interpretation from the corresponding rule probabilities.

Your explanation should make clear that **ViterbiParser selects the parse with the highest product of rule probabilities**.

# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**


## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

## 4. Graph Traversal — Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.

In [8]:
# --- spaCy dependency parsing tasks ---
# If spaCy or the model is missing, uncomment and run the following:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from spacy import displacy
from collections import deque

nlp = spacy.load("en_core_web_sm")

text = "The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation."
parsed_doc = nlp(text)

# 1) Find root
def find_root(doc):
    for token in doc:
        if token.dep_ == 'ROOT':
            return token
    return None

# 2) Extract terminals: start_node = 'researchers' (nsubj), end_node = 'mutation'
def extract_terminals(root, doc):
    start_node = None
    for child in root.children:
        if child.dep_ == 'nsubj':
            start_node = child
            break
    end_node = None
    for token in doc:
        if token.text.lower() == 'mutation':
            end_node = token
            break
    return start_node, end_node

# 3) BFS for shortest path in undirected dependency graph
def bfs_shortest_path(start, end):
    if start is None or end is None:
        return []
    q = deque()
    q.append((start, [start]))
    visited = {start}
    while q:
        node, path = q.popleft()
        if node == end:
            return path
        neighbors = list(node.children)
        if node.head is not node:
            neighbors.append(node.head)
        for nb in neighbors:
            if nb not in visited:
                visited.add(nb)
                q.append((nb, path + [nb]))
    return []

# 4) Path analysis printing
def print_path_details(path):
    print('| Word | POS | Head | Dependency |')
    print('|---|---|---|---|')
    for tok in path:
        head_text = tok.head.text if tok.head else ''
        print(f'| {tok.text} | {tok.pos_} | {head_text} | {tok.dep_} |')

root = find_root(parsed_doc)
start_node, end_node = extract_terminals(root, parsed_doc)
dependency_path = bfs_shortest_path(start_node, end_node)
print('Dependency path:', ' -> '.join([t.text for t in dependency_path]))
print_path_details(dependency_path)

# 6) Visual verification (in Jupyter):
# displacy.render(parsed_doc, style='dep', jupyter=True)


Dependency path: researchers -> discovered -> vaccine -> prevents -> mutation
| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| discovered | VERB | discovered | ROOT |
| vaccine | NOUN | discovered | dobj |
| prevents | VERB | vaccine | relcl |
| mutation | NOUN | prevents | dobj |
